# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset ([DOI: 10.71728/senscience.qs2f-h81p](https://sen.science/doi/10.71728/senscience.qs2f-h81p)) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema hosted at the following URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

This notebook will guide you through loading, inspecting, processing, and visualizing the data via its Croissant schema, referencing all entities by their `@id` fields.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading

We'll load the Croissant package metadata and its records using `mlcroissant`.
All objects in Croissant (record sets, fields, columns) will be referenced by their unique `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nVersion: {metadata.version}, Published: {metadata.datePublished}")

## 2. Data Overview

Let's examine which record sets, fields, and columns are available in the dataset, **by their `@id`**.

Most tabular Croissant datasets provide record sets describing the main table(s) and their fields, each with a unique `@id`.

In [ ]:
# List all record sets and their fields (@id)

# Collect record sets from the metadata, if available
croissant_json = dataset.metadata.to_json()
record_sets = croissant_json.get('recordSet', [])

if len(record_sets) == 0:
    print("No recordSet found in metadata!")
else:
    print("Available record sets and their fields:")
    for rset in record_sets:
        rset_id = rset.get('@id', None)
        name = rset.get('name', '')
        print(f"- RecordSet @id: {rset_id} (name: {name})")
        fields = rset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field @id: {field.get('@id', '?')} (name: {field.get('name','')})")
            else:
                print(f"    - Field @id: {field}")


The following cell lists a sample of the individual records from the first available record set so you can see typical data structures and field `@id`s.

In [ ]:
# Find the first record set's @id for inspection
if len(record_sets) == 0:
    print("No recordSet found.")
else:
    # Use the @id of the first record set
    first_record_set_id = record_sets[0]['@id']
    print(f"Sample records from RecordSet with @id: {first_record_set_id}")
    # Preview first 3 records
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(f"Record {i+1}: {record}")
        if i >= 2:
            break

## 3. Data Extraction

Let's load all the records from all record sets into Pandas DataFrames for convenient analysis.
**All references use the `@id`s returned above.**

In [ ]:
# Extract data from each record set into a DataFrame
dataframes = {}
for rset in record_sets:
    rset_id = rset['@id']
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Loaded RecordSet @id: {rset_id} ({len(df)} rows, {len(df.columns)} columns)")

Let's explore the first record set's columns and show the top few rows:

In [ ]:
if len(record_sets) > 0:
    first_rset_id = record_sets[0]['@id']
    print(f"Columns for RecordSet @id {first_rset_id}:")
    print(dataframes[first_rset_id].columns.tolist())
    display(dataframes[first_rset_id].head())
else:
    print("No recordSets available.")

## 4. Exploratory Data Analysis (EDA)

In this section, we'll demonstrate how to process tabular records using field and column `@id`s from the Croissant schema.

Typical EDA steps might include filtering by a numeric field (e.g., filtering patients by age), normalizing a numeric feature, or grouping by a key attribute (e.g., sex or anatomical site).

**All fields will be referenced by their `@id`.**

In [ ]:
# --- Example: Filter, normalize, and group by field @id ---

# We'll need to inspect the columns/fields to select candidates.
# For demonstration, suppose the 'Age_at_2nd_CRC' column exists;
# replace <Age_at_2nd_CRC_id> and <Sex_id> with the real @id from the earlier overview.

# Select record set
if len(record_sets) > 0:
    chosen_record_set = record_sets[0]
    chosen_rset_id = chosen_record_set['@id']
    df = dataframes[chosen_rset_id]
    
    # Attempt to find a likely numeric field (e.g., age)
    # We'll look for a column with 'age' or 'Age' in its name
    numeric_field_id = None
    for col in df.columns:
        if 'age' in str(col).lower():
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No field containing 'age' found; please inspect the columns above and adjust this demo.")
    else:
        print(f"Using numeric field @id: {numeric_field_id}")
        # Demonstration threshold
        threshold = 50
        # Filter
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records\n")
        display(filtered_df.head())
        
        # Normalize
        col_norm = numeric_field_id + "_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} (z-score):\n")
        display(filtered_df[[numeric_field_id, col_norm]].head())

        # Try grouping by a likely categorical field (e.g., 'sex', 'Sex')
        group_field_id = None
        for col in df.columns:
            if 'sex' in str(col).lower():
                group_field_id = col
                break
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:\n")
            display(grouped_df)
        else:
            print("No grouping field (e.g. 'Sex') found in columns; skipping grouping.")
else:
    print("No recordSets to demonstrate EDA.")

## 5. Visualization

Let's visualize one of the key variables (such as Age at 2nd CRC diagnosis, or MSI-status distributions) if available.

Adapt the field `@id` to match your schema's column.

In [ ]:
# Visualization: Histogram of the main numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets) > 0 and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True, color='dodgerblue')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
else:
    print("No numeric field to plot. Edit the field selection above to match your data.")

## 6. Conclusion

- This notebook demonstrated how to load, inspect, and process a dataset described by a Croissant schema via its `@id` fields, using Python and `mlcroissant`.
- All data extraction and references were made via the globally unique Croissant `@id` keys for true schema-level reproducibility.
- You can adapt the filtering, normalization, and visualization examples to your own analytic requirements by cross-referencing schema fields in Section 2 with actual data columns.

Explore further for richer analyses, statistical summaries, or cross-tabulations using pandas and visualization libraries.

*For more about Croissant, visit https://github.com/mlcommons/croissant and https://mlcommons.org/croissant/.*